# Epsilon-Greedy Algorithmus

Aufgabe: 

In [ ]:
import numpy as np
import pandas as pd

def epsilon_greedy_bandit(mus, T=25, eps=0.15, seed=42):
    """
    mus: echte Erfolgswahrscheinlichkeiten der Arme, z.B. [0.8, 0.5, 0.4]
    Reward ist Bernoulli: r ~ Bernoulli(mu_arm)
    """
    rng = np.random.default_rng(seed)
    K = len(mus)

    n = np.zeros(K, dtype=int)          # wie oft Arm i gewählt
    rewards_sum = np.zeros(K, dtype=int) # Summe Rewards je Arm
    mu_hat = np.full(K, 0.5, dtype=float)  # Startwert (wie auf Folie)

    rows = []

    for t in range(1, T + 1):
        # Exploration vs Exploitation
        u_explore = rng.random()
        explore = u_explore < eps

        if t <= K:
            # Warm-up: jeden Arm einmal spielen (t=1..K)
            arm = t - 1
            decision = "warmup"
        else:
            if explore:
                arm = rng.integers(0, K)   # zufälliger Arm
                decision = "explore"
            else:
                arm = int(np.argmax(mu_hat))  # bester bisher
                decision = "exploit"

        # Reward ziehen (Bernoulli)
        u_reward = rng.random()
        r = 1 if u_reward < mus[arm] else 0

        # Update
        n[arm] += 1
        rewards_sum[arm] += r
        mu_hat[arm] = rewards_sum[arm] / n[arm]

        rows.append({
            "t": t,
            "decision": decision,
            "u_explore": u_explore,
            "arm": arm + 1,  # 1..K für Menschen
            "u_reward": u_reward,
            "reward": r,
            "n1": n[0], "n2": n[1], "n3": n[2],
            "mu_hat1": mu_hat[0], "mu_hat2": mu_hat[1], "mu_hat3": mu_hat[2],
            "cum_reward": int(sum(rewards_sum))
        })

    return pd.DataFrame(rows)

df = epsilon_greedy_bandit([0.8, 0.5, 0.4], T=25, eps=0.15, seed=1)
df.head()
